# 2100年単収予測（ブートストラップ100回）

- 学習: 2014年までのデータ、同じXGBoostパラメータ
- 予測: future_2100 の2つのCSV（SSP245, SSP370）の気候を入力
- ブートストラップ100回 → 平均値・95%信頼区間下限を全ヶ国・全穀物で出力

In [1]:
import pandas as pd
import numpy as np
import os
from sklearn.metrics import r2_score
import xgboost as xgb

base_dir = r'C:\Users\xyz19\OneDrive\デスクトップ\予測モデルデータセット'
os.chdir(base_dir)

regressionDF = pd.read_excel(os.path.join(base_dir, 'regression_df_20260112_233447.xlsx'))
print(f"回帰データ: {regressionDF.shape}")
print(f"Year範囲: {regressionDF['Year'].min()} - {regressionDF['Year'].max()}")

回帰データ: (16388, 8)
Year範囲: 1961 - 2013


In [9]:
# 特徴量構築（load_regressionDF_simple と同じ）
climate_features = ['SPEI', 'GSL', 'Hurs', 'TXX']
area_dummies = pd.get_dummies(regressionDF['Area'], prefix='Area')
item_dummies = pd.get_dummies(regressionDF['Item'], prefix='Item')
year_feature = regressionDF[['Year']].copy()

X = pd.concat([
    regressionDF[climate_features],
    area_dummies,
    item_dummies,
    year_feature
], axis=1)
y = regressionDF['Yield'].copy()

train_mask = regressionDF['Year'] <= 2014
X_train = X.loc[train_mask]
y_train = y.loc[train_mask]

train_areas = sorted(regressionDF.loc[train_mask, 'Area'].unique().tolist())
train_items = sorted(regressionDF.loc[train_mask, 'Item'].unique().tolist())

best_params = {
    'colsample_bytree': 1.0,
    'learning_rate': 0.2,
    'max_depth': 5,
    'n_estimators': 300,
    'subsample': 0.9,
    'random_state': 42,
    'n_jobs': -1
}

print(f"訓練: Year<=2014, n={train_mask.sum()}")
print(f"Area数: {len(train_areas)}, Item数: {len(train_items)}")
print(f"X列数: {X.shape[1]}")

訓練: Year<=2014, n=16388
Area数: 156, Item数: 3
X列数: 164


In [10]:
def build_X_future(future_df, train_areas, train_items, feature_cols):
    """回帰データにある国（train_areas）だけを使い、future CSV の気候で X を構築（Year=2100）"""
    future_areas_set = set(future_df['Area'].unique())
    # 回帰にある国のみ。future にない国はスキップ
    areas_to_use = [a for a in train_areas if a in future_areas_set]
    
    rows = []
    for area in areas_to_use:
        r = future_df[future_df['Area'] == area].iloc[0]
        for item in train_items:
            rows.append({
                'Area': area, 'Item': item, 'Year': 2100,
                'SPEI': r['SPEI'], 'GSL': r['GSL'], 'Hurs': r['Hurs'], 'TXX': r['TXX']
            })
    
    if not rows:
        empty_X = pd.DataFrame(columns=feature_cols)
        return empty_X, pd.DataFrame(columns=['Area', 'Item'])
    
    df = pd.DataFrame(rows)
    
    area_cat = pd.Categorical(df['Area'], categories=train_areas)
    item_cat = pd.Categorical(df['Item'], categories=train_items)
    area_dum = pd.get_dummies(area_cat, prefix='Area')
    item_dum = pd.get_dummies(item_cat, prefix='Item')
    
    X_f = pd.concat([
        df[climate_features],
        area_dum,
        item_dum,
        df[['Year']]
    ], axis=1)
    
    for c in feature_cols:
        if c not in X_f.columns:
            X_f[c] = 0
    X_f = X_f[feature_cols]
    
    meta = df[['Area', 'Item']].copy()
    return X_f, meta

feature_cols = X_train.columns.tolist()
print("feature_cols 先頭10:", feature_cols[:10])

feature_cols 先頭10: ['SPEI', 'GSL', 'Hurs', 'TXX', 'Area_Afghanistan', 'Area_Albania', 'Area_Algeria', 'Area_Angola', 'Area_Antigua and Barbuda', 'Area_Argentina']


In [11]:
# future_2100 の2つのCSVを読み込み（国名は回帰データと完全一致のみ使用）
f370 = pd.read_csv(os.path.join(base_dir, 'future_2100_features_SSP370_bfuture_2096-2099_mean_20260129_043854.csv'))
f245 = pd.read_csv(os.path.join(base_dir, 'future_2100_features_SSP245_nfuture_2096-2099_mean_20260129_043901.csv'))

future_areas_370 = set(f370['Area'].unique())
future_areas_245 = set(f245['Area'].unique())
matched_370 = sorted([a for a in train_areas if a in future_areas_370])
matched_245 = sorted([a for a in train_areas if a in future_areas_245])
unmatched_370 = sorted([a for a in train_areas if a not in future_areas_370])
unmatched_245 = sorted([a for a in train_areas if a not in future_areas_245])

print("=== 国名マッチ状況（回帰 vs future CSV） ===")
print(f"回帰データの国数: {len(train_areas)}")
print(f"Future SSP370 の国数: {len(future_areas_370)}")
print(f"Future SSP245 の国数: {len(future_areas_245)}")
print()
print(f"【マッチした国】回帰にあり且つ future にあり（使用する）")
print(f"  SSP370: {len(matched_370)} ヶ国")
print(f"  SSP245: {len(matched_245)} ヶ国")
for i, a in enumerate(matched_370, 1):
    print(f"    {i:3d}. {a}")
print()
print(f"【マッチせず】回帰にあるが future にない（スキップ）")
print(f"  SSP370: {len(unmatched_370)} ヶ国")
print(f"  SSP245: {len(unmatched_245)} ヶ国")
for i, a in enumerate(unmatched_370, 1):
    print(f"    {i:3d}. {a}")
print("=" * 50)

X_future_370, meta_370 = build_X_future(f370, train_areas, train_items, feature_cols)
X_future_245, meta_245 = build_X_future(f245, train_areas, train_items, feature_cols)

n_countries = meta_370['Area'].nunique() if len(meta_370) > 0 else 0
print(f"\n対象: 回帰にある国のうち future に気候がある国のみ（{n_countries}ヶ国 × 3穀物）")
print(f"SSP370: {len(meta_370)} 行 (Area×Item)")
print(f"SSP245: {len(meta_245)} 行 (Area×Item)")
if len(meta_370) > 0:
    print(meta_370.head())

=== 国名マッチ状況（回帰 vs future CSV） ===
回帰データの国数: 156
Future SSP370 の国数: 246
Future SSP245 の国数: 246

【マッチした国】回帰にあり且つ future にあり（使用する）
  SSP370: 156 ヶ国
  SSP245: 156 ヶ国
      1. Afghanistan
      2. Albania
      3. Algeria
      4. Angola
      5. Antigua and Barbuda
      6. Argentina
      7. Armenia
      8. Australia
      9. Austria
     10. Azerbaijan
     11. Bangladesh
     12. Barbados
     13. Belarus
     14. Belgium
     15. Belize
     16. Benin
     17. Bhutan
     18. Bosnia and Herzegovina
     19. Botswana
     20. Brazil
     21. Brunei Darussalam
     22. Bulgaria
     23. Burkina Faso
     24. Burundi
     25. Cabo Verde
     26. Cambodia
     27. Cameroon
     28. Canada
     29. Central African Republic
     30. Chad
     31. Chile
     32. China
     33. Colombia
     34. Comoros
     35. Congo
     36. Costa Rica
     37. Croatia
     38. Cuba
     39. Cyprus
     40. Côte d'Ivoire
     41. Denmark
     42. Djibouti
     43. Dominica
     44. Dominican Republic
     4

In [12]:
N_BOOT = 100
rng = np.random.default_rng(42)
n_train = len(X_train)

if len(meta_370) == 0 or len(meta_245) == 0:
    raise ValueError("回帰にある国で、future に気候がある国がありません。future CSV の国名が回帰と完全一致するか確認してください。")

preds_370 = np.zeros((N_BOOT, len(meta_370)))
preds_245 = np.zeros((N_BOOT, len(meta_245)))

for b in range(N_BOOT):
    idx = rng.integers(0, n_train, size=n_train)
    X_b = X_train.iloc[idx]
    y_b = y_train.iloc[idx]
    model = xgb.XGBRegressor(**best_params)
    model.fit(X_b, y_b)
    preds_370[b] = model.predict(X_future_370)
    preds_245[b] = model.predict(X_future_245)
    if (b + 1) % 20 == 0:
        print(f" bootstrap {b+1}/{N_BOOT} 完了")

print("ブートストラップ100回 完了")

 bootstrap 20/100 完了
 bootstrap 40/100 完了
 bootstrap 60/100 完了
 bootstrap 80/100 完了
 bootstrap 100/100 完了
ブートストラップ100回 完了


In [13]:
def stats_by_area_item(preds, meta):
    mean_ = np.mean(preds, axis=0)
    ci95_lo = np.percentile(preds, 2.5, axis=0)
    df = meta.copy()
    df['mean_yield'] = mean_
    df['ci95_lower'] = ci95_lo
    return df

res_370 = stats_by_area_item(preds_370, meta_370)
res_245 = stats_by_area_item(preds_245, meta_245)

def overall_stats(preds):
    flat = preds.flatten()
    return np.mean(flat), np.percentile(flat, 2.5)

mean_all_370, ci_all_370 = overall_stats(preds_370)
mean_all_245, ci_all_245 = overall_stats(preds_245)

print("=== 全ヶ国・全穀物（SSP370） ===")
print(f"  平均単収: {mean_all_370:.2f} hg/ha")
print(f"  95%信頼区間下限: {ci_all_370:.2f} hg/ha")
print()
print("=== 全ヶ国・全穀物（SSP245） ===")
print(f"  平均単収: {mean_all_245:.2f} hg/ha")
print(f"  95%信頼区間下限: {ci_all_245:.2f} hg/ha")

=== 全ヶ国・全穀物（SSP370） ===
  平均単収: 4117.49 hg/ha
  95%信頼区間下限: 1037.58 hg/ha

=== 全ヶ国・全穀物（SSP245） ===
  平均単収: 4043.63 hg/ha
  95%信頼区間下限: 1000.49 hg/ha


In [14]:
print("=== 全ヶ国・全穀物 一覧（平均値・95%CI下限） ===")
summary = pd.DataFrame({
    'scenario': ['SSP370', 'SSP245'],
    'mean_yield_hg_per_ha': [mean_all_370, mean_all_245],
    'ci95_lower_hg_per_ha': [ci_all_370, ci_all_245]
})
print(summary.to_string(index=False))
print()
print("--- 国・穀物別（SSP370）先頭20行 ---")
print(res_370.head(20).to_string(index=False))
print()
print("--- 国・穀物別（SSP245）先頭20行 ---")
print(res_245.head(20).to_string(index=False))

=== 全ヶ国・全穀物 一覧（平均値・95%CI下限） ===
scenario  mean_yield_hg_per_ha  ci95_lower_hg_per_ha
  SSP370           4117.489924           1037.584067
  SSP245           4043.628877           1000.489850

--- 国・穀物別（SSP370）先頭20行 ---
               Area         Item  mean_yield  ci95_lower
        Afghanistan Maize (corn) 2732.766285 2207.543347
        Afghanistan         Rice 3392.264983 3005.075775
        Afghanistan        Wheat 2005.648293 1670.094244
            Albania Maize (corn) 5367.270186 4500.830908
            Albania         Rice 5619.973198 4872.939575
            Albania        Wheat 3275.294792 2785.499493
            Algeria Maize (corn) 5284.735148 1971.698987
            Algeria         Rice 3744.563441 1744.457864
            Algeria        Wheat 2733.696924  919.999930
             Angola Maize (corn)  963.235374  670.216463
             Angola         Rice 1272.601050  874.214917
             Angola        Wheat 1184.374538  914.377589
Antigua and Barbuda Maize (corn) 2400.87

In [16]:
res_370['scenario'] = 'SSP370'
res_245['scenario'] = 'SSP245'
out = pd.concat([res_370, res_245], ignore_index=True)
out = out[['scenario', 'Area', 'Item', 'mean_yield', 'ci95_lower']]
out.to_csv(os.path.join(base_dir, 'predict_2100_bootstrap_results.csv'), index=False)
print("保存: predict_2100_bootstrap_results.csv")
summary.to_csv(os.path.join(base_dir, 'predict_2100_bootstrap_summary.csv'), index=False)
print("保存: predict_2100_bootstrap_summary.csv")

保存: predict_2100_bootstrap_results.csv
保存: predict_2100_bootstrap_summary.csv
